In [0]:
# ----------------------------------------
# Notebook: 01_ingest_data
# Project: Retail Data Lakehouse
# Purpose:
# Load raw retail data and perform initial profiling.
# ----------------------------------------

df = spark.table("default.online_retail_ii")

In [0]:
display(df.limit(10))


In [0]:
df.printSchema()

In [0]:
print(f"Total Records: {df.count():,}")

In [0]:
from pyspark.sql.functions import col, count, when

null_counts = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
])

display(null_counts)

In [0]:
# Count duplicate rows
total_rows = df.count()
distinct_rows = df.distinct().count()

print(f"Total Rows    : {total_rows:,}")
print(f"Distinct Rows : {distinct_rows:,}")
print(f"Duplicate Rows: {total_rows - distinct_rows:,}")

In [0]:
%sql
SHOW SCHEMAS IN workspace;

In [0]:
df = spark.table("workspace.default.online_retail_ii")

In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
df.createOrReplaceTempView("online_retail")

In [0]:
%sql
SELECT *
FROM online_retail
LIMIT 10;

In [0]:
%sql
SELECT COUNT(*) AS total_rows
FROM online_retail;

In [0]:
%sql
SELECT DISTINCT Country
FROM online_retail
ORDER BY Country;

In [0]:
%sql
SELECT
    Country,
    COUNT(*) AS total_orders
FROM online_retail
GROUP BY Country
ORDER BY total_orders DESC;

In [0]:
clean_df = df.dropDuplicates()

In [0]:
clean_df.count()

In [0]:
df = spark.table("workspace.default.online_retail_ii")

In [0]:
silver_df = df.dropDuplicates()

In [0]:
print("Bronze:", df.count())
print("Silver:", silver_df.count())

In [0]:
from pyspark.sql.functions import col, sum

silver_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in silver_df.columns
]).show()

In [0]:
silver_df.filter(col("Description").isNull()).show(10, truncate=False)

In [0]:
df = spark.table("workspace.default.online_retail_ii")

In [0]:
df.columns

In [0]:
display(
    df.select("Customer ID", "Price")
)

In [0]:
customers_df = spark.read.csv(
    "/FileStore/tables/customers.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "/FileStore/tables/products.csv",
    header=True,
    inferSchema=True
)

orders_df = spark.read.csv(
    "/FileStore/tables/orders.csv",
    header=True,
    inferSchema=True
)

sales_df = spark.read.csv(
    "/FileStore/tables/sales.csv",
    header=True,
    inferSchema=True
)

In [0]:
df = spark.table("workspace.default.online_retail_ii")

df.printSchema()
df.show(5)

In [0]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

In [0]:
clean_df = df.dropna(subset=["Customer ID"])

clean_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in clean_df.columns
]).show()

In [0]:
clean_df = clean_df.withColumn(
    "Revenue",
    clean_df.Quantity * clean_df.Price
)

In [0]:
clean_df.select(
    "Quantity",
    "Price",
    "Revenue"
).show(5)

In [0]:
clean_df.groupBy("Customer ID") \
    .sum("Revenue") \
    .orderBy("sum(Revenue)", ascending=False)

In [0]:
top_customers = (
    clean_df
    .groupBy("Customer ID")
    .sum("Revenue")
    .orderBy("sum(Revenue)", ascending=False)
    .limit(10)
)

top_customers.show()

In [0]:
from pyspark.sql.functions import sum

top_customers = (
    clean_df
    .groupBy("Customer ID")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy("TotalRevenue", ascending=False)
    .limit(10)
)

top_customers.show()

In [0]:
from pyspark.sql.functions import sum, round

top_customers = (
    clean_df
    .groupBy("Customer ID")
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy("TotalRevenue", ascending=False)
    .limit(10)
)